In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#%matplotlib inline

#import seaborn as sns


import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
# from celloracle import motif_analysis as ma
# import celloracle as co


In [2]:
import glob as glob

In [3]:
networks = glob.glob('/data2st1/junyi/output/atac1112/cicre/*network.csv')
df_peaks = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/cCRE_annotated.csv')

In [ ]:
# peaks = df_peaks.names.str.replace("[:-]","_")
# tss_annotated = ma.get_tss_info(peak_str_list=peaks, ref_genome="mm10")


que bed peaks: 1667332
tss peaks in que: 45233


In [ ]:
df_peaks_promoter = df_peaks[df_peaks['primary_region']=='promoter']
df_peaks_promoter['Peakp'] = df_peaks_promoter['names'].str.replace("[:-]","_")

In [61]:
enhancers = pd.DataFrame()
for network in networks:
    df_cire_tmp = pd.read_csv(network,index_col=0)
    df_cire_selected = df_cire_tmp[df_cire_tmp.pval_adj<0.05]
    #df_cire_selected = df_cire_selected.rename({"score":"coaccess"},axis=1)

    df_enh1 = df_cire_selected.merge(df_peaks_promoter[['Peakp','gene_name']],left_on='Peak1',right_on='Peakp',how='inner')[['Peak2','Peakp','score','gene_name']]
    df_enh1.columns = ['peak_id','promoter','coaccess','gene_short_name']
    df_enh2 = df_cire_selected.merge(df_peaks_promoter[['Peakp','gene_name']],left_on='Peak2',right_on='Peakp',how='inner')[['Peak1','Peakp','score','gene_name']]
    df_enh2.columns = ['peak_id','promoter','coaccess','gene_short_name']
    integrated_enhancer = pd.concat([df_enh1,df_enh2],axis=0)
    integrated_enhancer.sort_values(by='coaccess',ascending=False,inplace=True)
    integrated_enhancer.drop_duplicates(subset='peak_id', inplace=True)

    if df_cire_selected.shape[0]==0:
        continue
    # integrated = ma.integrate_tss_peak_with_cicero(tss_peak=tss_annotated,
    #                                             cicero_connections=df_cire_selected)
    # integrated_enhancer = integrated[integrated.coaccess<1]
    integrated_enhancer_split = integrated_enhancer.peak_id.str.split("_",expand=True)
    integrated_enhancer['names'] = integrated_enhancer_split[0] + ":" + integrated_enhancer_split[1].astype(str) + "-" + integrated_enhancer_split[2].astype(str)
    region_subclass = network.split('/')[-1].replace('_ALL_circe_network.csv','').replace('HIP','HPF').replace('AMY_AMY','AMY').replace('HPF_HPF','HPF').replace('PFC_PFC','PFC')
    integrated_enhancer['Region Subclass'] = region_subclass
    integrated_enhancer.to_csv(network.replace('_ALL_circe_network.csv','_enhancers.csv'))
    enhancers = pd.concat([enhancers,integrated_enhancer],axis=0)

In [4]:
fenhancers = glob.glob('/data2st1/junyi/output/atac1112/cicre/*enhancers.csv')
enhancers = pd.DataFrame()
for enhancer in fenhancers:
    df_tmp = pd.read_csv(enhancer,index_col=0)
    enhancers = pd.concat([enhancers,df_tmp],axis=0)

In [5]:
enhancers

,peak_id,promoter,coaccess,gene_short_name,names,Region Subclass
1,chr18_83185522_83186023,chr18_83399371_83399872,0.908645,Gm50413,chr18:83185522-83186023,AMY_MOL-1
0,chr1_171713668_171714169,chr1_171710750_171711251,0.853936,Tma7-ps,chr1:171713668-171714169,AMY_MOL-1
34,chr7_83857882_83858383,chr7_83872953_83873454,0.850116,Gm44530,chr7:83857882-83858383,AMY_MOL-1
38,chr2_19368695_19369196,chr2_19369259_19369760,0.841803,Msrb2,chr2:19368695-19369196,AMY_MOL-1
69,chr18_86921617_86922118,chr18_86927209_86927710,0.837851,Gm5971,chr18:86921617-86922118,AMY_MOL-1
...,...,...,...,...,...,...
26578,chr3_36202901_36203402,chr3_36000461_36000962,0.136311,Mccc1,chr3:36202901-36203402,HPF_Mossy_Glut
31923,chr13_67011640_67012141,chr13_67360387_67360888,0.136309,Gm17039,chr13:67011640-67012141,HPF_Mossy_Glut
32285,chr3_54457467_54457968,chr3_54155242_54155743,0.136307,Trpc4,chr3:54457467-54457968,HPF_Mossy_Glut
3867,chr8_122867660_122868161,chr8_123236030_123236531,0.136304,Gm45842,chr8:122867660-122868161,HPF_Mossy_Glut


In [3]:
df_tobias = pd.read_parquet('/data2st1/junyi/output/atac1112/tobias/annotated_filtered.parquet')

In [6]:
set(enhancers['Region Subclass']).difference(set(df_tobias['Region Subclass']))

{'AMY_Arachnoid_Barrier_cell',
 'AMY_Ependymal_cell',
 'AMY_MOL-3',
 'AMY_NFOL',
 'AMY_Perivascular_Macrophage',
 'AMY_VLMC',
 'HPF_Astrocyte-3',
 'HPF_COP',
 'HPF_Ependymal_cell',
 'HPF_Immature_cell',
 'HPF_Subiculum_CT_Glut',
 'HPF_VLMC',
 'PFC_Astrocyte-3',
 'PFC_COP',
 'PFC_MOL-3',
 'PFC_Perivascular_Macrophage',
 'PFC_Pvalb_Vipr2_GABA',
 'PFC_VLMC'}

In [7]:
set(df_tobias['Region Subclass']).difference(set(enhancers['Region Subclass']))


{'HPF_OPC'}

In [8]:
df_tobias_enhancers = pd.merge(df_tobias,enhancers,on=['names','Region Subclass'],how='left')

In [9]:
df_tobias_enhancers.head()

,TFBS_chr,TFBS_start,TFBS_end,TFBS_name,TFBS_score,TFBS_strand,peak_chr,peak_start,peak_end,MC_score,...,primary_region,secondary_region,encodeCCRE,Sex,Ensemble,Gene,peak_id,promoter,coaccess,gene_short_name
0,chr1,3309945,3309954,metacluster_137.2cisbp__M01001_None,9.25971,-,chr1,3309937,3310438,0.11586,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
1,chr1,3309945,3309954,metacluster_137.2cisbp__M01001_None,9.34861,-,chr1,3309937,3310438,0.36816,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
2,chr1,3309945,3309954,metacluster_137.2cisbp__M01001_None,9.26355,-,chr1,3309937,3310438,0.46053,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
3,chr1,3310134,3310143,metacluster_68.2cisbp__M00407_None,7.61715,+,chr1,3309937,3310438,0.20840,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
4,chr1,3309944,3309955,metacluster_137.2transfac_public__M00173_None,7.82886,-,chr1,3309937,3310438,0.11586,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN


In [10]:
df_tobias_enhancers_candidate = df_tobias_enhancers[df_tobias_enhancers.coaccess.notnull()]

In [11]:
import re
import numpy as np
import pandas as pd

_interval_re = re.compile(r"^(chr[^_]+)_(\d+)_(\d+)$")

def _parse_interval_str(s: str):
    """Parse 'chr1_3309937_3310438' -> (chrom, start, end); invalid -> (None, None, None)."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return (None, None, None)
    m = _interval_re.match(str(s).strip())
    if not m:
        return (None, None, None)
    chrom = m.group(1)
    start = int(m.group(2))
    end = int(m.group(3))
    if start > end:
        start, end = end, start
    return (chrom, start, end)

def add_pedistance(df: pd.DataFrame, peak_col: str = "peak_id", promoter_col: str = "promoter") -> pd.DataFrame:
    """
    In-place add column 'pedistance' = nearest boundary gap between df[peak_col] and df[promoter_col].

    Rules:
      - Different chromosome -> inf
      - Overlap or touch -> 0
      - Else -> start_right - end_left
      - Invalid interval string -> NaN
    """
    peak_parsed = df[peak_col].map(_parse_interval_str)
    prom_parsed = df[promoter_col].map(_parse_interval_str)

    peak_chr = peak_parsed.map(lambda x: x[0])
    peak_start = peak_parsed.map(lambda x: x[1])
    peak_end = peak_parsed.map(lambda x: x[2])

    prom_chr = prom_parsed.map(lambda x: x[0])
    prom_start = prom_parsed.map(lambda x: x[1])
    prom_end = prom_parsed.map(lambda x: x[2])

    pedist = pd.Series(np.nan, index=df.index, dtype="float64")

    valid = peak_chr.notna() & prom_chr.notna()

    # Different chromosomes -> inf
    pedist.loc[valid & (peak_chr != prom_chr)] = np.inf

    # Same chromosome -> compute nearest boundary gap
    same = valid & (peak_chr == prom_chr)
    ps = peak_start[same].to_numpy()
    pe = peak_end[same].to_numpy()
    rs = prom_start[same].to_numpy()
    re_ = prom_end[same].to_numpy()

    # Ensure left interval is the one with smaller start
    left_end = np.where(ps <= rs, pe, re_)
    right_start = np.where(ps <= rs, rs, ps)

    pedist.loc[same] = np.where(right_start <= left_end, 0.0, (right_start - left_end).astype(float))

    df["pedistance"] = pedist
    return df


In [12]:
df_tobias_enhancers_candidate = add_pedistance(df_tobias_enhancers_candidate, peak_col='peak_id', promoter_col='promoter')

/tmp/ipykernel_3635406/718569337.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pedistance"] = pedist


In [ ]:
promoter_prox = df_tobias_enhancers_candidate[(df_tobias_enhancers_candidate.primary_region!='promoter')&(df_tobias_enhancers_candidate.pedistance<=200)]
promoter_prox['PEstatus'] = 'promoter_proximal'
near_prox = df_tobias_enhancers_candidate[(df_tobias_enhancers_candidate.primary_region.isin(['distal','intron']))&\
                                          (df_tobias_enhancers_candidate.pedistance>200)&(df_tobias_enhancers_candidate.pedistance<=2000)]
near_prox['PEstatus'] = 'near_proximal_enhancer'
distal = df_tobias_enhancers_candidate[(df_tobias_enhancers_candidate.primary_region.isin(['distal','intron']))&(df_tobias_enhancers_candidate.pedistance>2000)&(df_tobias_enhancers_candidate.pedistance<=500_000)]
distal['PEstatus'] = 'distal_enhancer'
addcolumns = ['promoter','coaccess','gene_short_name','pedistance','PEstatus']
for addcol in addcolumns:
    df_tobias[addcol] = np.nan
df_tobias[df_tobias.primary_region=='promoter']['PEstatus'] = 'promoter'
for df, name in zip([promoter_prox, near_prox, distal],
                    ['promoter_proximal_enhancers','near_promoter_enhancers','distal_enhancers']):
    print(f"Saving {name} with shape {df.shape}")

    # 确保按 index 对齐写回
    df_tobias.loc[df.index, addcolumns] = df.loc[df.index, addcolumns]

/tmp/ipykernel_3635406/3927001270.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  promoter_prox['PEstatus'] = 'promoter_proximal'


In [19]:
df_tobias.to_parquet('/data2st1/junyi/output/atac1112/tobias/annotated_filtered_with_enhancers.parquet')

In [166]:
df_tobias_raw = pd.read_parquet('/data2st1/junyi/output/atac1112/tobias/annotated_filtered.parquet')